# Задание

Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

In [5]:
import pyspark.sql.functions as F
from pyspark.sql import Row
from pyspark.sql import SparkSession

In [6]:
spark = SparkSession.builder.getOrCreate()
spark

In [8]:
df = spark.read.format("xml").option("rowTag", "row").load('/content/posts_sample.xml')

df.show()

+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|    _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|               _Tags|              _Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+----------

In [9]:
languages_df = spark.read.format('csv').option('header', 'true').option("inferSchema", True).load('/content/programming-languages.csv').dropna()

languages_df.show()

+------------+--------------------+
|        name|       wikipedia_url|
+------------+--------------------+
|     A# .NET|https://en.wikipe...|
|  A# (Axiom)|https://en.wikipe...|
|  A-0 System|https://en.wikipe...|
|          A+|https://en.wikipe...|
|         A++|https://en.wikipe...|
|        ABAP|https://en.wikipe...|
|         ABC|https://en.wikipe...|
|   ABC ALGOL|https://en.wikipe...|
|       ABSET|https://en.wikipe...|
|       ABSYS|https://en.wikipe...|
|         ACC|https://en.wikipe...|
|      Accent|https://en.wikipe...|
|    Ace DASL|https://en.wikipe...|
|        ACL2|https://en.wikipe...|
|     ACT-III|https://en.wikipe...|
|     Action!|https://en.wikipe...|
|ActionScript|https://en.wikipe...|
|         Ada|https://en.wikipe...|
|     Adenine|https://en.wikipe...|
|        Agda|https://en.wikipe...|
+------------+--------------------+
only showing top 20 rows


In [10]:
# сначала фиьтруем по годам

dates = ("2010-01-01",  "2020-12-31")
posts_by_date = df.filter(F.col("_CreationDate").between(*dates))
posts_by_date.show(5)

+-----------------+------------+--------------------+-----------+-------------+-------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+-----+------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount|_CommunityOwnedDate|       _CreationDate|_FavoriteCount|    _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|_Tags|_Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+-------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+-----+------+----------+
|             NULL|        NULL|<p>No. (And more ...|       NULL|  

In [11]:
# фильтруем по языкам

language_names = [str(x[0]) for x in languages_df.collect()]

def includes_name(x):
    tag = None
    for name in language_names:
        n = '<' + name.lower() + '>'
        if n in str(x._Tags).lower():
            tag = name
            break
    if tag is None:
        tag = 'No'

    return (x[6], tag)

posts_by_date_rdd = posts_by_date.rdd.map(includes_name).filter(lambda x: x[1] != 'No')

posts_by_date_rdd_group = posts_by_date_rdd.keyBy(lambda row: (row[0].year, row[1])).aggregateByKey(0, lambda x, y: x + 1, lambda x1, x2: x1 + x2).sortBy(lambda x: x[1], ascending=False).collect()

years_list = [i for i in range(2010, 2020)][::-1]
df_by_years = []
for year in years_list:
    df_by_years.extend([row for row in posts_by_date_rdd_group if row[0][0] == year][:10])

row_template = Row('Year', 'Language', 'Count')
result_df = spark.createDataFrame([row_template(*x, y) for x, y in df_by_years])

In [12]:
result_df.show(100)

+----+-----------+-----+
|Year|   Language|Count|
+----+-----------+-----+
|2019|     Python|  162|
|2019| JavaScript|  131|
|2019|       Java|   95|
|2019|        PHP|   59|
|2019|          R|   36|
|2019|          C|   14|
|2019|     MATLAB|    9|
|2019|         Go|    9|
|2019|       Dart|    9|
|2019|       Bash|    8|
|2018|     Python|  214|
|2018| JavaScript|  196|
|2018|       Java|  145|
|2018|        PHP|   99|
|2018|          R|   63|
|2018|          C|   24|
|2018|      Scala|   22|
|2018| TypeScript|   21|
|2018| PowerShell|   13|
|2018|       Bash|   12|
|2017| JavaScript|  244|
|2017|       Java|  204|
|2017|     Python|  185|
|2017|        PHP|  122|
|2017|          R|   53|
|2017|          C|   24|
|2017|Objective-C|   19|
|2017|       Ruby|   16|
|2017| PowerShell|   14|
|2017| TypeScript|   14|
|2016| JavaScript|  272|
|2016|       Java|  179|
|2016|     Python|  141|
|2016|        PHP|  126|
|2016|          R|   50|
|2016|          C|   32|
|2016|       Ruby|   21|


In [13]:
# сохраняем

result_df.write.mode("overwrite").parquet("result")